In [1]:
from typing import Iterator

import os
import string
import numpy as np
import pandas as pd
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

## Create synthetic dataset

In [2]:
UNITS = {"KB": 10**3, "MB": 10**6, "GB": 10**9}  # 1 base = 1 byte as ASCII

def parse_size(size: str) -> int:
    """'500KB', '1.5MB' or '2GB' -> number of bases."""
    number, unit = size[:-2], size[-2:].upper()
    if unit not in UNITS:
        raise ValueError(f"Invalid size: {size!r}. Use KB, MB or GB, e.g. '500KB', '1.5MB', '2GB'.")
    return int(float(number) * UNITS[unit])


def random_contig(size: str | int, seed: int | None = None) -> dict:
    """size is '500KB'/'1.5MB'/'2GB', or a number of bases."""
    rng = np.random.default_rng(seed)
    n_bases = parse_size(size) if isinstance(size, str) else size

    bases = np.frombuffer(b"ACGT", dtype=np.uint8)
    sequence = bases[rng.integers(0, 4, size=n_bases, dtype=np.uint8)].tobytes().decode("ascii")

    genome_id = f"GCF_{rng.integers(10**9):09d}.1"                                         # e.g. GCF_004512387.1
    contig_id = "".join(rng.choice(list(string.ascii_uppercase), 2)) + f"{rng.integers(10**6):06d}.1"  # e.g. OQ832096.1
    host = rng.choice(["Eco", "Sau", "Pae", "Kpn", "Asp", "Hme"])
    phage = f"vB_{host}{rng.choice(['M', 'P', 'S'])}_{rng.choice(['Kalo', 'Mira', 'Tove', 'Zeno'])}{rng.integers(1, 100)}"

    return {
        "genome_id": genome_id,
        "contig_id": contig_id,
        "sequence": sequence,
        "orientation": "forward",
        "description": f"{contig_id} Synthetic phage {phage}, complete genome",
    }

In [3]:
contig = random_contig("1KB", seed=0)

In [4]:
contig

{'genome_id': 'GCF_272657960.1',
 'contig_id': 'GF888118.1',
 'sequence': 'CGTTTTAGATTGTGACGTTCGACAATCAGAAAACTACCATCACGCCGTCGTGCCCGCGGTCCTGGCTGTAAGTGCGATCTCAACCTTTGGGGCCGATTTCGTCTAGTGCAGACTTTAGTGGAGTCTTAACTAACTTACGATAGGAAGAATGCTTACTAACAGACATAACTCACATAGTAAGAGGGTGGAAGGACTCTTGGCAGTAGACATTCCACTATATCAATACACTGCGATATTCGGGGATGAAGGCAGGCGCGAATAGGAAAAGGCGGACCTTGCGACACTACCAACCTGCCCGAGGCGTGGGATCATGAAGCCGCTGAGCTCGTTACCACCGTAGTAAGCGAGGCGCGGGTGACCTCACAATTGGGCAAAAGATGCAGGTGCAGAGAGAGCGAATCTGCCCCTCTACTCCAAAGTCCCTCTCAGCTAGTTGGTACGGTGAGCAAGATTCCCGCATCTTTCCGGACAAGGCTGCCAGGTGGGGCGAATATTATCCTCGAAAAGGTGGGACATGCCCACTTATAGTATTGCAATAATACATATCAATCCTGTGAGAAGGGGCAAGGGGTAACTGCTCGGGCATCCTACTATAAAATCCTTACTTCGAATACTGCAGCAGGTCTGGATTTCCAAGGTCAGGTGGTTTCCCAACCATGCCAGGTCTACCATCGTATTACCAACGAAGGGTGGCAGCTGTGCGTAACAGACAGCTAAGGGGACTGGCTTTAGTATTTAATTTTAGCGACGATACAAAGATCTTTTGATCAACTCAACACGGATTCCACATTTATCCCAGTGGTAACTCGAGCGTCACAGTGACTTGTATCCTCGGTAAACTCACTGTGGCAGTACCTAAAGACTTTAGTTCAACCTATCGTTTAGGCGGGCGTTCAAACTCTAGAGCCGAAACTCAGAATGAAAT

In [5]:
len(contig["sequence"])

1000

In [6]:
def random_contig_lengths(
    n_contigs: int,
    total_size: str,
    giant_fraction: float = 0.05,
    giant_factor: float = 50.0,
    seed: int | None = None,
) -> np.ndarray:
    """Lengths of n_contigs contigs whose sum is at most total_size.

    Lengths vary lognormally, and giant_fraction of the contigs (at least one) are on
    average giant_factor times longer than the rest.
    """
    total = parse_size(total_size)
    if not 0 < n_contigs <= total:
        raise ValueError(f"Need 0 < n_contigs <= total bases, got {n_contigs} contigs for {total:,} bases.")
    rng = np.random.default_rng(seed)

    weights = rng.lognormal(mean=0.0, sigma=1.0, size=n_contigs)
    n_giants = max(1, round(giant_fraction * n_contigs))
    weights[rng.choice(n_contigs, size=n_giants, replace=False)] *= giant_factor

    # every contig gets at least 1 base; the rest is shared by weight and rounded down,
    # so the sum never exceeds the total
    return 1 + np.floor(weights / weights.sum() * (total - n_contigs)).astype(np.int64)


In [7]:
random_contig_lengths(10, "1MB", seed=0)

array([ 10594,   8186, 886201,  10375,   5468,  13412,  34416,  24085,
         4622,   2636])

In [8]:
def random_contigs(
    n_contigs: int,
    total_size: str,
    giant_fraction: float = 0.05,
    giant_factor: float = 50.0,
    seed: int | None = None,
) -> Iterator[dict]:
    """Yields n_contigs random contigs with a total length of at most total_size.

    Contigs are generated one at a time, so only one sequence is in memory unless you
    keep them, e.g. with list(...).
    """
    rng = np.random.default_rng(seed)
    lengths = random_contig_lengths(n_contigs, total_size, giant_fraction, giant_factor, seed=rng.integers(2**63))
    for length in lengths.tolist():
        yield random_contig(length, seed=rng.integers(2**63))

In [9]:
contigs = list(random_contigs(1000, "100MB", seed=0))

In [10]:
ds = pd.DataFrame(contigs)

In [11]:
ds.head()

,genome_id,contig_id,sequence,orientation,description
0,GCF_096233192.1,EP190191.1,CTAATTGCGGCGCGGTAGCTATGAGGCCAACGGGATATAATGCGCC...,forward,"EP190191.1 Synthetic phage vB_KpnS_Zeno40, com..."
1,GCF_684951774.1,NL179703.1,TACTCTGGCCCGTAATCTGAAATGCAGTGCTTTTTGAAGGTGAAGG...,forward,"NL179703.1 Synthetic phage vB_HmeP_Tove15, com..."
2,GCF_612223867.1,MS008687.1,CGGCATAATTGCCTTAGCCGGTTACCTTCAATTTCACCAAGGAGAG...,forward,"MS008687.1 Synthetic phage vB_AspS_Mira47, com..."
3,GCF_141609484.1,FK296361.1,GCGTTTCTCGTGCGTCACAAACAGGGCGCCCGGGCTTCAGTCAGGC...,forward,"FK296361.1 Synthetic phage vB_PaeM_Tove73, com..."
4,GCF_956603096.1,VB270098.1,ATTCCAGGATTCGGTTTCTCCAGAGGCGATAGTGTTTTATATCGAA...,forward,"VB270098.1 Synthetic phage vB_EcoM_Zeno58, com..."


In [12]:
def save_synthetic_data(df: pd.DataFrame, file_path: str = "synthetic_data.fasta") -> None:
    records = []
    for _, row in df.iterrows():
        records.append(SeqRecord(
            Seq(row["sequence"]),
            id=str(row["contig_id"]),
            description=f"genome_id={row['genome_id']}, orientation={row['orientation']}, {row['description']}",
        ))
    SeqIO.write(records, file_path, "fasta")

### Save different synthetic dataset sizes

In [13]:
base_dir = "/home/datasets/experiments/experiment-bioinfo"

In [14]:
# n, size = 1_000, "100MB"
# contigs = list(random_contigs(n, size, seed=0))
# ds = pd.DataFrame(contigs)
# save_synthetic_data(ds, f"{base_dir}/synthetic_data_{n}_{size}.fasta")

In [15]:
# n, size = 10_000, "1GB"
# contigs = list(random_contigs(n, size, seed=0))
# ds = pd.DataFrame(contigs)
# save_synthetic_data(ds, f"{base_dir}/synthetic_data_{n}_{size}.fasta")

In [16]:
# n, size = 100_000, "10GB"
# contigs = list(random_contigs(n, size, seed=0))
# ds = pd.DataFrame(contigs)
# save_synthetic_data(ds, f"{base_dir}/synthetic_data_{n}_{size}.fasta")

## Stress test

### Sequence dataset

In [17]:
os.environ["PROKBERT_PROFILE"] = "1"

In [18]:
from prokbert.sequence_dataset import SequenceDataset

In [19]:
base_dir = '/home/datasets/experiments/experiment-bioinfo'

In [20]:
file_path_100MB = f"{base_dir}/synthetic_data_1000_100MB.fasta"
file_path_1GB = f"{base_dir}/synthetic_data_10000_1GB.fasta"
file_path_10GB = f"{base_dir}/synthetic_data_100000_10GB.fasta"

assert all(os.path.exists(fp) for fp in [file_path_100MB, file_path_1GB, file_path_10GB])

In [21]:
seq_ds = SequenceDataset()

In [22]:
seq_ds.create_dataset([file_path_100MB], save_dir=f"{base_dir}/test_save_dir1")

Loaded contigs from /home/datasets/experiments/experiment-bioinfo/synthetic_data_1000_100MB.fasta, 100.5 MB in 0.72 s -> 140.0 MB/s
Created dataset, 100.6 MB in 0.79 s -> 127.5 MB/s
Saved dataset to '/home/datasets/experiments/experiment-bioinfo/test_save_dir1' (100.4 MB) in 0.28 s -> 363.9 MB/s


In [23]:
seq_ds.load_dataset(f"{base_dir}/test_save_dir1")

Loaded dataset from '/home/datasets/experiments/experiment-bioinfo/test_save_dir1' (100.4 MB) in 0.04 s -> 2277.3 MB/s


In [24]:
seq_ds.create_dataset([file_path_1GB], save_dir=f"{base_dir}/test_save_dir2")

Loaded contigs from /home/datasets/experiments/experiment-bioinfo/synthetic_data_10000_1GB.fasta, 1005.0 MB in 4.77 s -> 210.8 MB/s
Created dataset, 1006.3 MB in 5.39 s -> 186.8 MB/s
Saved dataset to '/home/datasets/experiments/experiment-bioinfo/test_save_dir2' (1003.6 MB) in 2.34 s -> 429.2 MB/s


In [25]:
seq_ds.load_dataset(f"{base_dir}/test_save_dir2")

Loaded dataset from '/home/datasets/experiments/experiment-bioinfo/test_save_dir2' (1003.6 MB) in 0.22 s -> 4601.3 MB/s


In [26]:
seq_ds.create_dataset(
    [file_path_10GB], save_dir=f"{base_dir}/test_save_dir3"
)

Loaded contigs from /home/datasets/experiments/experiment-bioinfo/synthetic_data_100000_10GB.fasta, 10049.7 MB in 49.32 s -> 203.8 MB/s
Created dataset, 10062.8 MB in 55.37 s -> 181.7 MB/s
Saved dataset to '/home/datasets/experiments/experiment-bioinfo/test_save_dir3' (10036.3 MB) in 23.26 s -> 431.5 MB/s


In [27]:
seq_ds.load_dataset(f"{base_dir}/test_save_dir3")

Loaded dataset from '/home/datasets/experiments/experiment-bioinfo/test_save_dir3' (10036.3 MB) in 1.50 s -> 6672.9 MB/s


### Sampler Datasets

In [28]:
import time
from itertools import islice
from typing import Iterable, Iterator

from prokbert.types import Segment
from prokbert.dataset import RandomSegmentDataset, ContiguousSegmentDataset

In [29]:
def measure_segments(segments: Iterable[Segment], label: str = "measure segments throughput: ") -> Iterator[Segment]:
    """Pass segments through unchanged and print throughput at the end.

    Only time spent producing segments is counted.
    """
    seconds, n_bases, n_segments = 0.0, 0, 0
    iterator = iter(segments)
    try:
        while True:
            t0 = time.perf_counter()
            segment = next(iterator, None)
            seconds += time.perf_counter() - t0
            if segment is None:
                return
            start, end = segment["absolute_coordinate"]
            n_bases += end - start
            n_segments += 1
            yield segment
    finally:
        if n_segments and seconds > 0:
            print(
                f"{label}: {n_segments} segments, {n_bases / 1e6:.1f} Mbp in {seconds:.2f} s -> "
                f"{n_bases / 1e6 / seconds:.1f} Mbp/s, {n_segments / seconds:.0f} segments/s"
            )

In [30]:
contiguous_dataset = ContiguousSegmentDataset(
    sequence_dataset=seq_ds,
    min_segment_length=1_000,
    max_segment_length=2_000,
    return_sequence=False
)

for segment in measure_segments(contiguous_dataset, "measure contiguous segments throughput: "):
    pass

measure contiguous segments throughput: : 5000085 segments, 9975.0 Mbp in 7.32 s -> 1363.3 Mbp/s, 683357 segments/s


In [31]:
contiguous_dataset = ContiguousSegmentDataset(
    sequence_dataset=seq_ds,
    min_segment_length=1_000,
    max_segment_length=2_000,
    return_sequence=True
)

for segment in measure_segments(contiguous_dataset, "measure contiguous segments throughput with return seq: "):
    pass

measure contiguous segments throughput with return seq: : 5000085 segments, 9975.0 Mbp in 28.70 s -> 347.5 Mbp/s, 174205 segments/s


In [32]:
random_dataset = RandomSegmentDataset(
    sequence_dataset=seq_ds,
    min_segment_length=1_000,
    max_segment_length=2_000,
    return_sequence=False
)

for segment in measure_segments(islice(random_dataset, 1_000_000), "measure random segments throughput: "):
    pass

measure random segments throughput: : 1000000 segments, 1999.9 Mbp in 3.03 s -> 659.4 Mbp/s, 329710 segments/s


In [33]:
random_dataset = RandomSegmentDataset(
    sequence_dataset=seq_ds,
    min_segment_length=1_000,
    max_segment_length=2_000,
    return_sequence=True
)

for segment in measure_segments(islice(random_dataset, 1_000_000), "measure random segments throughput with return seq: "):
    pass

measure random segments throughput with return seq: : 1000000 segments, 1999.9 Mbp in 6.18 s -> 323.7 Mbp/s, 161843 segments/s
